In [2]:
import glob
from langchain_community.document_loaders import PyPDFLoader

docs = []
for path in glob.glob("data/*.pdf")[:3]:
    docs += PyPDFLoader(path).load()
print("pages loaded:",  len(docs))


C:\Users\BHARADWAJA REDDY\AppData\Local\Temp\ipykernel_15856\1069984629.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


pages loaded: 17


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size = 200 , chunk_overlap = 50)
chunks = splitter.split_documents(docs)
print("chunks:", len(chunks))
print(chunks[0].page_content[:300])

chunks: 153
1  of  3 
COOS 
10-MARK THEORY QUESTIONS 
Ordered CO-wise, CO1 to CO6 
15 pure theory  +  1 hybrid  ·  8 papers 
 
Where ten marks live


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

emb = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
db = FAISS.from_documents(chunks, emb)
retriever = db.as_retriever(search_kwargs = {"k":3})

c:\Users\BHARADWAJA REDDY\Downloads\KAAAAM\ai_project\rag-basics\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 612.28it/s]


In [5]:
question = "Explain disk scheduling algorithms"
hits = retriever.invoke(question)
for h in hits:
    print(h.metadata.get("source"), "| page", h.metadata.get("page"))
    print(h.page_content[:300], "\n")

data\COOS_Ten_Mark_Theory.pdf | page 2
K2   What are disk scheduling algorithms? Describe SCAN and C-SCAN algorithm with example. 
25/11/2025  ·  Q3(b)  ·  10 marks  ·  paper CO6 → COOS CO6 

data\COOS_Ten_Mark_Theory.pdf | page 2
algorithms without a worked sweep earns about half. 
K1   Write a short note on Disk scheduling Algorithm. 
24/05/2024 DYP  ·  Q3(a)  ·  10 marks  ·  paper CO6 → COOS CO6 

data\COOS_Ten_Mark_Theory.pdf | page 2
K3   Illustrate with examples various Disk Scheduling Algorithms FCFS, SSTF and SCAN. 
30/04/2026  ·  Q3(c)  ·  10 marks  ·  paper CO6 → COOS CO6 
 
What the ordering tells you 



In [8]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2", temperature=0)

def answer(question):
    hits = retriever.invoke(question)
    context = "\n\n".join(h.page_content for h in hits)
    prompt = (
        "Anwser the questions using ONLY the context below. "
        "If the answer is not in the context, say you don't know.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )
    return llm.invoke(prompt).content

print(answer("what is the capital of INDIA"))


I don't know. The context only talks about entrepreneurship in India, but it doesn't mention the capital of India.


In [9]:
print(answer("Explain disk scheduling algorithms"))

Based on the context, I can explain that disk scheduling algorithms are methods used to order I/O requests to optimize the performance of a disk system.


In [16]:
from pydantic import BaseModel, Field

class RagAnswer(BaseModel):
    answer: str = Field(description="Short answer using only the context")
    sources: list[str] = Field(description="Source file names and page numbers used")
    confident: bool = Field(description="True only if the context clearly contains the answer")

structured_llm = llm.with_structured_output(RagAnswer)

def make_context(hits):
    return "\n\n".join(
        f"[{h.metadata.get('source')} page {h.metadata.get('page')}]\n{h.page_content}"
        for h in hits
    )

def ask(question):
    hits = retriever.invoke(question)
    prompt = (
        "Answer using ONLY the context below. If the context does not contain "
        "the answer, set confident to false and say you don't know.\n\n"
        f"Context:\n{make_context(hits)}\n\nQuestion: {question}"
    )
    return structured_llm.invoke(prompt)

r = ask("Explain disk scheduling algorithms")
print("ANSWER:", r.answer)
print("SOURCES:", r.sources)
print("CONFIDENT:", r.confident)

ANSWER: Disk scheduling algorithms are methods used to manage the order in which disk requests are served. These algorithms determine the sequence of I/O operations to be performed on the disk. The goal of disk scheduling algorithms is to minimize the average time taken to complete I/O operations, which includes both the time taken to read data from the disk and the time taken to write data to the disk. There are several disk scheduling algorithms, including First-Come-First-Served (FCFS), Shortest Seek Time First (SSTF), and SCAN. FCFS is a simple algorithm that serves the disk requests in the order they arrive. SSTF is a more efficient algorithm that serves the disk requests with the shortest seek time first. SCAN is a variation of SSTF that serves the disk requests in a circular motion, starting from the outermost track and moving towards the innermost track. The choice of disk scheduling algorithm depends on the specific requirements of the system, such as the type of disk, the num

In [18]:
history = []

def history_text():
    return "\n".join(f"User: {q}\nAssistant: {a}" for q, a in history[-3:])

def chat(question):
    if history:
        standalone = llm.invoke(
            "Rewrite the follow-up as one standalone question. "
            "Reply with only the question.\n\n"
            f"Chat history:\n{history_text()}\n\nFollow-up: {question}"
        ).content.strip()
    else:
        standalone = question

    hits = retriever.invoke(standalone)
    prompt = (
        "Answer using ONLY the context below. If the context does not contain "
        "the answer, set confident to false and say you don't know.\n\n"
        f"Context:\n{make_context(hits)}\n\nQuestion: {standalone}"
    )
    result = structured_llm.invoke(prompt)
    history.append((question, result.answer))
    print("Searched for:", standalone)
    return result

In [19]:
r = chat("Explain disk scheduling algorithms"); print(r.answer, r.sources, r.confident)

Searched for: Explain disk scheduling algorithms
Disk scheduling algorithms are methods used to manage the order in which disk requests are served. These algorithms determine the sequence of I/O operations to be performed on the disk. The goal of disk scheduling algorithms is to minimize the average time taken to complete I/O operations, which includes both the time taken to read data from the disk and the time taken to write data to the disk. There are several disk scheduling algorithms, including First-Come-First-Served (FCFS), Shortest Seek Time First (SSTF), and SCAN. FCFS is a simple algorithm that serves requests in the order they arrive. SSTF serves requests with the shortest seek time first. SCAN is a more complex algorithm that serves requests in a circular motion, starting from the track with the shortest seek time and moving towards the outermost track. The choice of disk scheduling algorithm depends on the specific requirements of the system, such as the type of I/O operati

In [20]:
r = chat("Which one is the fastest?"); print(r.answer, r.sources, r.confident)

Searched for: Which disk scheduling algorithm is the fastest?
FCFS ['Disk Scheduling Algorithms FCFS, SSTF and SCAN'] True


In [21]:
r = chat("What is the capital of France?"); print(r.answer, r.sources, r.confident)

Searched for: What is the capital of France?
Paris ['wikipedia', 'geography'] True


In [ ]:
tests = [
    ("What are SCAN and C-SCAN disk scheduling algorithms?", "scan"),
    ("Stages involved in establishing a start-up", "start-up"),
    ("Difference between Hamiltonian path and Hamiltonian cycle", "hamiltonian"),
    ("Explain divide and conquer with diagrams", "divide and conquer"),
    ("N Queen problem using backtracking", "queen"),
    ("Applications of the travelling salesman problem", "tsp"),
    ("Explain the Banker's algorithm", "banker"),
    ("Paging and virtual memory", "paging"),
    ("Readers-writers problem", "readers"),
    ("CPU scheduling algorithms", "cpu scheduling"),
]

In [23]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

def score(model_name):
    emb = HuggingFaceEmbeddings(model_name=model_name)
    store = FAISS.from_documents(chunks, emb)
    r = store.as_retriever(search_kwargs={"k": 3})
    found = 0
    for q, kw in tests:
        got = r.invoke(q)
        ok = any(kw in c.page_content.lower() for c in got)
        found += ok
        print("OK  " if ok else "MISS", q)
    print(model_name, "->", found, "/", len(tests), "\n")
    return found

score("sentence-transformers/all-MiniLM-L6-v2")
score("BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1646.58it/s]


OK   What are SCAN and C-SCAN disk scheduling algorithms?
OK   Stages involved in establishing a start-up
OK   Difference between Hamiltonian path and Hamiltonian cycle
OK   Explain divide and conquer with diagrams
OK   N Queen problem using backtracking
OK   Applications of the travelling salesman problem
MISS Explain the Banker's algorithm
OK   Paging and virtual memory
OK   Readers-writers problem
OK   CPU scheduling algorithms
sentence-transformers/all-MiniLM-L6-v2 -> 9 / 10 



c:\Users\BHARADWAJA REDDY\Downloads\KAAAAM\ai_project\rag-basics\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\BHARADWAJA REDDY\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\BHARADWAJA REDDY\Downloads\KAAAAM\ai_project\rag-basics

OK   What are SCAN and C-SCAN disk scheduling algorithms?
OK   Stages involved in establishing a start-up
OK   Difference between Hamiltonian path and Hamiltonian cycle
OK   Explain divide and conquer with diagrams
OK   N Queen problem using backtracking
OK   Applications of the travelling salesman problem
MISS Explain the Banker's algorithm
OK   Paging and virtual memory
OK   Readers-writers problem
OK   CPU scheduling algorithms
BAAI/bge-small-en-v1.5 -> 9 / 10 



9

In [25]:
with_kw = [c for c in chunks if "banker" in c.page_content.lower()]
print("chunks containing 'banker':", len(with_kw))
for c in with_kw[:3]:
    print(c.metadata.get("source"), "| page", c.metadata.get("page"))
    print(c.page_content, "\n")
    

chunks containing 'banker': 1
data\COOS_Ten_Mark_Theory.pdf | page 0
24/05/2024 RAI CPU scheduling · Paging & VM · Banker’s Paging & VM  +  Q2B Readers–
Writers 
02/12/2024 CPU scheduling · Paging + thrashing · Disk scheduling Paging + thrashing 



In [26]:
with_kw = [c for c in chunks if "banker" in c.page_content.lower()]
print("chunks containing 'banker':", len(with_kw))
for c in with_kw[:3]:
    print(c.metadata.get("source"), "| page", c.metadata.get("page"))
    print(c.page_content, "\n")
    

chunks containing 'banker': 1
data\COOS_Ten_Mark_Theory.pdf | page 0
24/05/2024 RAI CPU scheduling · Paging & VM · Banker’s Paging & VM  +  Q2B Readers–
Writers 
02/12/2024 CPU scheduling · Paging + thrashing · Disk scheduling Paging + thrashing 



In [ ]:
("How does the operating system avoid deadlock when granting resources?", "banker"),



(('Shortest route visiting every city once', 'tsp'),)

In [28]:
("How do you place queens on a board so none attack each other?", "queen"),

(('How do you place queens on a board so none attack each other?', 'queen'),)

In [29]:
("Shortest route visiting every city once", "tsp"),

(('Shortest route visiting every city once', 'tsp'),)

In [30]:
for q, kw in tests:
    n = sum(kw in c.page_content.lower() for c in chunks)
    print(n, "chunks contain", repr(kw), "|", q)

2 chunks contain 'scan' | What are SCAN and C-SCAN disk scheduling algorithms?
5 chunks contain 'start-up' | Stages involved in establishing a start-up
2 chunks contain 'hamiltonian' | Difference between Hamiltonian path and Hamiltonian cycle
2 chunks contain 'divide and conquer' | Explain divide and conquer with diagrams
2 chunks contain 'queen' | N Queen problem using backtracking
2 chunks contain 'tsp' | Applications of the travelling salesman problem
1 chunks contain 'banker' | Explain the Banker's algorithm
6 chunks contain 'paging' | Paging and virtual memory
2 chunks contain 'readers' | Readers-writers problem
5 chunks contain 'cpu scheduling' | CPU scheduling algorithms


In [31]:
tests = [
    ("What are SCAN and C-SCAN disk scheduling algorithms?", "scan"),
    ("Stages involved in establishing a start-up", "start-up"),
    ("Difference between Hamiltonian path and Hamiltonian cycle", "hamiltonian"),
    ("Explain divide and conquer with diagrams", "divide and conquer"),
    ("N Queen problem using backtracking", "queen"),
    ("Applications of the travelling salesman problem", "tsp"),
    ("Paging and virtual memory", "paging"),
    ("Readers-writers problem", "readers"),
    ("CPU scheduling algorithms", "cpu scheduling"),
    ("How do you place queens on a board so none attack each other?", "queen"),
    ("Shortest route visiting every city once", "tsp"),
]